# Progetto: Timed GNN per la risoluzione di puzzle scacchistici (Lichess)
### Fase 1 — Pipeline Dati

Questo notebook copre la **Fase 1: Data Preparation** del progetto:

1. Download del database puzzle di Lichess
2. Caricamento ed esplorazione
3. Filtro dei puzzle "mate in n" (n = 1..5, con estensione a 10 per il held-out set)
4. Validazione di FEN e mosse con `python-chess`
5. Simulazione delle feature temporali (i puzzle Lichess NON hanno timing reale)
6. Split 80/10/10 stratificato per `n`
7. Salvataggio dei dataset
8. Struttura per il set di validazione "classico" (held-out, esterno a Lichess)

> **Nota importante sulla libreria custom**: il testo del progetto cita una libreria di Wong/Damiani per l'addestramento, a cui al momento non abbiamo accesso. Questo notebook produce dati in un formato standard (CSV/Parquet + grafi PyTorch Geometric nella fase successiva) così da restare compatibile sia con quella libreria (se recuperata) sia con un'implementazione standard in PyTorch Geometric.

**Ambiente consigliato:** Google Colab, runtime con GPU (Runtime > Change runtime type > GPU) per le fasi successive di training.


## 0. Setup — installazione dipendenze

In [ ]:
# Installazione librerie necessarie (Colab)
!pip install -q python-chess zstandard pyarrow tqdm scikit-learn requests


In [ ]:
import os
import io
import json
import zstandard as zstd
import pandas as pd
import numpy as np
import chess
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

tqdm.pandas()

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)


## 1. Download del database puzzle Lichess

Il database ufficiale dei puzzle è disponibile come CSV compresso in `.zst` su `database.lichess.org`.
Contiene oltre 2 milioni di puzzle con colonne:

`PuzzleId, FEN, Moves, Rating, RatingDeviation, Popularity, NbPlays, Themes, GameUrl, OpeningTags`

**Attenzione ai campi `FEN` e `Moves`**: la posizione `FEN` è quella *prima* che l'avversario giochi la propria mossa. La prima mossa in `Moves` è quindi la mossa dell'avversario che genera la posizione del puzzle; le mosse successive sono la **soluzione** che il giocatore (o il nostro modello) deve trovare. Bisogna tenerne conto quando si costruisce il grafo/etichetta.


In [ ]:
import requests

PUZZLE_URL = "https://database.lichess.org/lichess_db_puzzle.csv.zst"
ZST_PATH = os.path.join(DATA_DIR, "lichess_db_puzzle.csv.zst")
CSV_PATH = os.path.join(DATA_DIR, "lichess_db_puzzle.csv")

def download_file(url, path, chunk_size=1024 * 1024):
    if os.path.exists(path):
        print(f"File gia' presente: {path}")
        return
    print(f"Download da {url} ...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                pbar.update(len(chunk))
    print("Download completato.")

download_file(PUZZLE_URL, ZST_PATH)


In [ ]:
# Decompressione .zst -> .csv
def decompress_zst(zst_path, out_path):
    if os.path.exists(out_path):
        print(f"CSV gia' presente: {out_path}")
        return
    dctx = zstd.ZstdDecompressor()
    with open(zst_path, "rb") as ifh, open(out_path, "wb") as ofh:
        dctx.copy_stream(ifh, ofh)
    print("Decompressione completata.")

decompress_zst(ZST_PATH, CSV_PATH)


## 2. Caricamento ed esplorazione

Il file completo è ~250-300MB e oltre 2 milioni di righe: per iniziare carichiamo tutto in Pandas
(gestibile in RAM su Colab), ma teniamo pronta un'opzione a chunk nel caso servisse ridurre la memoria.


In [ ]:
COLUMNS = ["PuzzleId", "FEN", "Moves", "Rating", "RatingDeviation",
           "Popularity", "NbPlays", "Themes", "GameUrl", "OpeningTags"]

df = pd.read_csv(CSV_PATH, names=COLUMNS, header=0)
print(f"Totale puzzle: {len(df):,}")
df.head()


In [ ]:
df[["Rating", "Popularity", "NbPlays"]].describe()


## 3. Filtro dei puzzle "mate in n"

I temi (`Themes`) includono etichette come `mateIn1`, `mateIn2`, ..., `mateIn5` (raramente oltre).
Estraiamo `n` direttamente dal tema e scartiamo i puzzle che non sono di matto puro (es. puzzle tattici
che non terminano in scacco matto), per restare coerenti con l'obiettivo "mate in n moves".


In [ ]:
import re

MATE_THEME_RE = re.compile(r"^mateIn(\d+)$")

def extract_mate_n(themes_str):
    if not isinstance(themes_str, str):
        return None
    for theme in themes_str.split():
        m = MATE_THEME_RE.match(theme)
        if m:
            return int(m.group(1))
    return None

df["mate_n"] = df["Themes"].apply(extract_mate_n)
df_mate = df[df["mate_n"].notna()].copy()
df_mate["mate_n"] = df_mate["mate_n"].astype(int)

print(f"Puzzle di tipo 'mate in n': {len(df_mate):,} su {len(df):,} totali")
df_mate["mate_n"].value_counts().sort_index()


In [ ]:
# Per il training ci concentriamo su n = 1..5 (come da metodologia del progetto)
df_train_pool = df_mate[df_mate["mate_n"].between(1, 5)].copy()
print(f"Puzzle mate in 1-5 disponibili per il training set: {len(df_train_pool):,}")
df_train_pool["mate_n"].value_counts().sort_index()


## 4. Validazione di FEN e mosse con `python-chess`

Verifichiamo che ogni FEN sia una posizione legale e che la sequenza di mosse UCI sia interamente
legale a partire da quella posizione. Scartiamo eventuali righe corrotte. Verifichiamo inoltre che il
numero di mosse "di soluzione" (escludendo la prima mossa dell'avversario) sia coerente con `mate_n`
(una mossa di matto in n richiede n mosse del giocatore + n-1 mosse di risposta dell'avversario, più la
mossa iniziale dell'avversario che genera la posizione).


In [ ]:
def validate_puzzle(row):
    try:
        board = chess.Board(row["FEN"])
    except ValueError:
        return False, None

    moves = row["Moves"].split()
    if len(moves) < 2:
        return False, None

    try:
        for uci in moves:
            move = chess.Move.from_uci(uci)
            if move not in board.legal_moves:
                return False, None
            board.push(move)
    except (ValueError, AssertionError):
        return False, None

    is_checkmate = board.is_checkmate()
    # mosse di soluzione = tutte tranne la prima (mossa dell'avversario che apre il puzzle)
    solution_len_player_moves = (len(moves) - 1 + 1) // 2  # arrotondato per eccesso
    return True, is_checkmate

tqdm.pandas(desc="Validazione puzzle")
results = df_train_pool.progress_apply(validate_puzzle, axis=1)
df_train_pool["is_valid"] = results.apply(lambda x: x[0])
df_train_pool["ends_in_checkmate"] = results.apply(lambda x: x[1])

n_before = len(df_train_pool)
df_train_pool = df_train_pool[df_train_pool["is_valid"] & df_train_pool["ends_in_checkmate"]].copy()
n_after = len(df_train_pool)
print(f"Puzzle validi e che terminano in scacco matto: {n_after:,} / {n_before:,}")


## 5. Simulazione delle feature temporali

Il dataset puzzle di Lichess **non contiene timing reale** (a differenza del database delle partite
"elite", che invece ha i tempi per mossa nel PGN). Come indicato nella metodologia, simuliamo un tempo di
pensiero per mossa basandoci su:

- **Rating del puzzle**: puzzle più difficili -> tempo di pensiero simulato più alto
- **Popularity/NbPlays**: puzzle molto giocati tendono ad avere soluzioni più "intuitive" -> leggero
  sconto sul tempo

La funzione produce un tempo in secondi per la mossa di soluzione, che verrà poi usato come feature di
nodo/arco nella costruzione del grafo (fase successiva).


In [ ]:
def simulate_think_time(rating, popularity, base=3.0, rating_scale=400.0):
    """Simula il tempo di pensiero (in secondi) per una mossa di soluzione.

    - rating alto  -> puzzle piu' difficile -> piu' tempo
    - popularity alta -> puzzle piu' 'intuitivo' per la community -> leggero sconto
    """
    rating = float(rating) if pd.notna(rating) else 1500.0
    popularity = float(popularity) if pd.notna(popularity) else 0.0

    difficulty_factor = max(rating - 1000.0, 0.0) / rating_scale
    popularity_discount = np.clip(popularity / 100.0, -1.0, 1.0) * 0.5

    think_time = base + difficulty_factor - popularity_discount
    # rumore per evitare valori identici a parita' di rating/popularity
    think_time += np.random.normal(0, 0.5)
    return float(max(think_time, 0.5))

np.random.seed(42)
df_train_pool["simulated_think_time_sec"] = df_train_pool.apply(
    lambda r: simulate_think_time(r["Rating"], r["Popularity"]), axis=1
)

df_train_pool[["Rating", "Popularity", "mate_n", "simulated_think_time_sec"]].sample(10, random_state=1)


## 6. Split train/val/test (80/10/10), stratificato per `n`

Lo split interno serve per l'addestramento e la validazione del modello (metodologia, sezione 1).
Il set di validazione "classico" (fase 8 sotto) è **separato** e va usato solo per la valutazione finale
GNN vs LLM, mai per il training.


In [ ]:
train_frames, val_frames, test_frames = [], [], []

for n_val, group in df_train_pool.groupby("mate_n"):
    train_g, temp_g = train_test_split(group, test_size=0.20, random_state=42)
    val_g, test_g = train_test_split(temp_g, test_size=0.50, random_state=42)
    train_frames.append(train_g)
    val_frames.append(val_g)
    test_frames.append(test_g)

df_train = pd.concat(train_frames).reset_index(drop=True)
df_val = pd.concat(val_frames).reset_index(drop=True)
df_test = pd.concat(test_frames).reset_index(drop=True)

print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")
print("\nDistribuzione per n (train):")
print(df_train['mate_n'].value_counts().sort_index())


## 7. Salvataggio dei dataset (Parquet, più compatto e veloce di CSV per gli step successivi)

In [ ]:
KEEP_COLS = ["PuzzleId", "FEN", "Moves", "Rating", "Popularity",
             "mate_n", "simulated_think_time_sec"]

df_train[KEEP_COLS].to_parquet(os.path.join(DATA_DIR, "puzzles_train.parquet"), index=False)
df_val[KEEP_COLS].to_parquet(os.path.join(DATA_DIR, "puzzles_val.parquet"), index=False)
df_test[KEEP_COLS].to_parquet(os.path.join(DATA_DIR, "puzzles_test.parquet"), index=False)

print("Salvati in:", DATA_DIR)
!ls -lh {DATA_DIR}


## 8. Set di validazione "classico" (held-out, esterno a Lichess)

Come da metodologia, il set di validazione finale per il confronto GNN vs LLM deve venire da **fuori**
Lichess (es. dataset Kaggle "Chess Puzzles", o database puzzle di Chess.com), con `n` che copre un
range più ampio (1-10) per testare i limiti di profondità.

Questa cella definisce solo la **struttura attesa** e un caricatore: scarica manualmente uno dei
dataset (es. da Kaggle, serve un account/API token) e carica il file con questa cella. Il formato
target è identico a quello dei file Parquet sopra, così le fasi successive (costruzione grafo, training,
valutazione) trattano train/val/test e held-out in modo uniforme.


In [ ]:
# ESEMPIO: se hai scaricato un CSV con colonne tipo FEN, Solution/Moves, Rating, Themes
# (adatta i nomi colonna al dataset che scegli, es. Kaggle 'Chess Puzzles' di lichess-puzzles)

HELD_OUT_CSV = os.path.join(DATA_DIR, "held_out_classic_puzzles.csv")  # <- carica qui il tuo file

if os.path.exists(HELD_OUT_CSV):
    df_held_out = pd.read_csv(HELD_OUT_CSV)
    print(f"Held-out set caricato: {len(df_held_out):,} puzzle")
    # TODO: applicare qui le stesse funzioni di validazione (sezione 4) e
    # rinominare le colonne per allinearle a KEEP_COLS, poi:
    # df_held_out.to_parquet(os.path.join(DATA_DIR, "puzzles_held_out.parquet"), index=False)
else:
    print(
        "Nessun file held-out trovato.\n"
        f"Carica un dataset esterno (es. Kaggle 'Chess Puzzles') come '{HELD_OUT_CSV}'\n"
        "e ri-esegui questa cella. In Colab puoi anche usare:\n"
        "  from google.colab import files\n"
        "  uploaded = files.upload()"
    )


## Riepilogo Fase 1 e prossimi passi

**Output prodotti in `/content/data/`:**
- `puzzles_train.parquet`, `puzzles_val.parquet`, `puzzles_test.parquet` — split interni stratificati per `mate_n`, con FEN validati, mosse verificate, e `simulated_think_time_sec` come proxy del timing
- struttura pronta per `puzzles_held_out.parquet` (dataset esterno, per il confronto finale GNN vs LLM)

**Prossimi passi (fasi successive del progetto):**
1. **Costruzione del grafo**: trasformare ogni FEN in un grafo PyTorch Geometric (nodi = caselle/pezzi, archi = mosse legali/attacchi/difese), con `simulated_think_time_sec` come feature di arco/nodo
2. **Modello GNN "timed"**: implementazione in PyTorch Geometric (policy head + eventuale value head), in assenza della libreria Wong/Damiani
3. **Baseline LLM**: prompting su un modello open (es. Llama) o via API per lo stesso set di puzzle
4. **Valutazione comparativa**: accuracy per valore di `n`, McNemar's test, grafici

Fammi sapere quando vuoi passare alla costruzione del grafo (punto 1) e procedo con il prossimo notebook.
